# EZStats — Event Spotter: SN-BAS-2025 Training
Train a Bi-LSTM on SoccerNet Ball Action Spotting 2025 labels.
Uses ResNet18 features extracted locally — consistent with inference in `event_spotter.py`.

**Run order: A → B → C → D (inspect!) → E → F (update classes!) → G → H → I**

## Cell A — Install + Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q huggingface_hub torch torchvision

## Cell B — Download SN-BAS-2025 from HuggingFace
You need a HuggingFace account + token from huggingface.co/settings/tokens.
~19GB total — downloads labels + encrypted video zips to Drive.

In [ ]:
from huggingface_hub import snapshot_download, login

login()  # paste your HF token when prompted

LOCAL_DIR = '/content/drive/MyDrive/ezstats/sn-bas-2025'

snapshot_download(
    repo_id='SoccerNet/SN-BAS-2025',
    repo_type='dataset',
    revision='main',
    local_dir=LOCAL_DIR,
)
print('Download complete:', LOCAL_DIR)

## Cell C — Extract encrypted video zips
SoccerNet password: `s0cc3rn3t`

In [ ]:
import zipfile
from pathlib import Path

SN_BAS_ROOT = Path('/content/drive/MyDrive/ezstats/sn-bas-2025')
PASSWORD = b's0cc3rn3t'

zip_files = sorted(SN_BAS_ROOT.rglob('*.zip'))
print(f'Found {len(zip_files)} zip files')

for zf in zip_files:
    out_dir = zf.parent
    try:
        with zipfile.ZipFile(zf) as z:
            z.extractall(out_dir, pwd=PASSWORD)
        print(f'  Extracted: {zf.name}')
    except Exception as e:
        print(f'  SKIP {zf.name}: {e}')

print('Extraction complete.')

## Cell D — Inspect label format
**Run this before Cell F.** Check the actual class names and position format, then update Cell F accordingly.

In [ ]:
import json
from pathlib import Path

SN_BAS_ROOT = Path('/content/drive/MyDrive/ezstats/sn-bas-2025')

label_files = sorted(SN_BAS_ROOT.rglob('*.json'))
print(f'Found {len(label_files)} JSON files')
for f in label_files[:5]:
    print(' ', f.relative_to(SN_BAS_ROOT))

if label_files:
    data = json.loads(label_files[0].read_text())
    print('\nTop-level keys:', list(data.keys()))
    anns = data.get('annotations', data.get('events', []))
    print(f'Annotation count: {len(anns)}')
    if anns:
        print('First 3 annotations:')
        for a in anns[:3]:
            print(' ', a)

# Collect all unique class labels
all_labels = set()
for f in label_files:
    try:
        d = json.loads(f.read_text())
        for a in d.get('annotations', d.get('events', [])):
            lbl = a.get('label', a.get('type', ''))
            if lbl:
                all_labels.add(lbl)
    except:
        pass
print('\nAll unique action labels:', sorted(all_labels))

if label_files and anns:
    print('\nSample positions:', [a.get('position') for a in anns[:5]])
    print('Sample gameTimes:', [a.get('gameTime') for a in anns[:5]])

# List video files found
videos = sorted(SN_BAS_ROOT.rglob('*.mkv')) + sorted(SN_BAS_ROOT.rglob('*.mp4'))
print(f'\nVideo files found: {len(videos)}')
for v in videos[:6]:
    print(' ', v.relative_to(SN_BAS_ROOT))

## Cell E — Extract ResNet18 features from videos
Saves `*.resnet18_2fps.npy` next to each video in Drive. Skip-safe (won't re-extract).

In [ ]:
import cv2, numpy as np, torch
import torch.nn as nn
from pathlib import Path
from torchvision import models, transforms

SN_BAS_ROOT = Path('/content/drive/MyDrive/ezstats/sn-bas-2025')
FEAT_FPS = 2.0  # must match event_spotter.py FEAT_FPS

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
backbone.fc = nn.Identity()
backbone = backbone.to(device).eval()

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_features(video_path, feat_fps=2.0):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    interval = max(1, int(round(fps / feat_fps)))
    features, frame_idx = [], 0
    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % interval == 0:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                t = preprocess(rgb).unsqueeze(0).to(device)
                feat = backbone(t).squeeze(0).cpu().numpy()
                norm = np.linalg.norm(feat)
                features.append(feat / norm if norm > 0 else feat)
            frame_idx += 1
    cap.release()
    return np.array(features, dtype=np.float32), fps

video_files = sorted(SN_BAS_ROOT.rglob('*.mkv')) + sorted(SN_BAS_ROOT.rglob('*.mp4'))
print(f'Found {len(video_files)} videos')

for video_path in video_files:
    feat_path = video_path.with_suffix('.resnet18_2fps.npy')
    fps_path  = video_path.with_suffix('.fps.txt')
    if feat_path.exists():
        print(f'  SKIP: {video_path.name}')
        continue
    print(f'  Extracting {video_path.name}...', end='', flush=True)
    feats, source_fps = extract_features(video_path)
    np.save(str(feat_path), feats)
    fps_path.write_text(str(source_fps))
    print(f' {len(feats)} frames @ {source_fps:.1f}fps → saved')

print('Feature extraction done.')

## Cell F — Build dataset
**UPDATE `BALL_ACTION_CLASSES` based on Cell D output before running this.**

In [ ]:
import json, numpy as np
from pathlib import Path
from collections import Counter

SN_BAS_ROOT = Path('/content/drive/MyDrive/ezstats/sn-bas-2025')
FEAT_FPS    = 2.0
WINDOW      = 15

# ⚠️ UPDATE after Cell D shows actual class names
BALL_ACTION_CLASSES = [
    'background',
    'PASS',
    'DRIVE',
    'HEADER',
    'HIGH PASS',
    'OUT',
    'CORNER',
    'CROSS',
    'THROW IN',
    'SHOT',
    'BALL PLAYER BLOCK',
    'PLAYER SUCCESSFUL TACKLE',
    'FREE KICK',
    'GOAL',
]
CLASS_TO_IDX = {c.upper(): i for i, c in enumerate(BALL_ACTION_CLASSES)}
NUM_CLASSES  = len(BALL_ACTION_CLASSES)
print(f'{NUM_CLASSES} classes:', BALL_ACTION_CLASSES)

def position_to_feat_frame(ann, source_fps, feat_fps=2.0):
    pos = ann.get('position', 0)
    # if position > 100000 → milliseconds; else → frames at source_fps
    if pos > 100000:
        return int(pos / 1000 * feat_fps)
    else:
        return int(pos / source_fps * feat_fps)

def parse_half(gametime_str):
    try:
        return int(gametime_str.split(' - ')[0].strip())
    except:
        return -1

def build_samples(feat_path, fps_path, label_path, half,
                  window=15, event_radius=1, bg_keep=0.04):
    feats      = np.load(str(feat_path))
    source_fps = float(fps_path.read_text().strip()) if fps_path.exists() else 25.0
    T          = len(feats)
    label_arr  = np.zeros(T, dtype=np.int64)

    data = json.loads(label_path.read_text())
    anns = data.get('annotations', data.get('events', []))
    for ann in anns:
        if parse_half(ann.get('gameTime', '')) != half:
            continue
        label_str = ann.get('label', ann.get('type', '')).upper()
        cls = CLASS_TO_IDX.get(label_str, 0)
        if cls == 0:
            continue
        ff = position_to_feat_frame(ann, source_fps)
        for k in range(max(0, ff - event_radius), min(T, ff + event_radius + 1)):
            label_arr[k] = cls

    samples = []
    for i in range(T - window + 1):
        center = label_arr[i + window // 2]
        if center == 0 and np.random.rand() > bg_keep:
            continue
        samples.append((feats[i:i+window].copy(), int(center)))
    return samples

np.random.seed(42)

label_files  = sorted(SN_BAS_ROOT.rglob('*.json'))
valid_dirs   = sorted(set(
    lf.parent for lf in label_files
    if list(lf.parent.glob('*.resnet18_2fps.npy'))
))
print(f'Game dirs with labels + features: {len(valid_dirs)}')

split      = max(1, int(len(valid_dirs) * 0.75))
train_dirs = valid_dirs[:split]
val_dirs   = valid_dirs[split:]
print(f'Train: {len(train_dirs)} games, Val: {len(val_dirs)} games')

def collect(dirs):
    samples = []
    for d in dirs:
        lf = list(d.glob('*.json'))
        if not lf:
            continue
        label_path = lf[0]
        for feat_path in sorted(d.glob('*.resnet18_2fps.npy')):
            half     = 1 if feat_path.name.startswith('1') else 2
            fps_path = feat_path.with_suffix('').with_suffix('.fps.txt')
            s = build_samples(feat_path, fps_path, label_path, half)
            samples.extend(s)
            print(f'  {d.name[:40]} h{half}: {len(s)} samples')
    return samples

train_samples = collect(train_dirs)
val_samples   = collect(val_dirs)

counts = Counter(s[1] for s in train_samples)
print('\nClass distribution (train):')
for i, name in enumerate(BALL_ACTION_CLASSES):
    if counts[i]:
        print(f'  {i:2d}  {name:<28} {counts[i]:>6}')
print(f'\nTrain: {len(train_samples)}, Val: {len(val_samples)}')

## Cell G — Train Bi-LSTM

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class EventDataset(Dataset):
    def __init__(self, s):
        self.X = torch.tensor(np.array([x[0] for x in s]), dtype=torch.float32)
        self.y = torch.tensor([x[1] for x in s], dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

total   = len(train_samples)
weights = [min(total / (NUM_CLASSES * max(counts[i], 1)), 50.0) for i in range(NUM_CLASSES)]
w_tensor = torch.tensor(weights, dtype=torch.float32).to(device)
print('Class weights:')
for i, (name, w) in enumerate(zip(BALL_ACTION_CLASSES, weights)):
    print(f'  {name:<28} {w:.1f}')

train_loader = DataLoader(EventDataset(train_samples), batch_size=256, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(EventDataset(val_samples),   batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

class EventSpotter(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.lstm = nn.LSTM(512, 256, batch_first=True, bidirectional=True, num_layers=2, dropout=0.3)
        self.head = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, n)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, out.shape[1] // 2])

model     = EventSpotter(NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(weight=w_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5, verbose=True)

best_val_loss, best_state = float('inf'), None

for epoch in range(1, 40 + 1):
    model.train()
    tl = tc = tt = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        tl += loss.item() * len(y)
        tc += (logits.argmax(1) == y).sum().item()
        tt += len(y)

    model.eval()
    vl = vc = vt = 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            vl += criterion(logits, y).item() * len(y)
            vc += (logits.argmax(1) == y).sum().item()
            vt += len(y)

    scheduler.step(vl / vt)
    flag = ''
    if vl / vt < best_val_loss:
        best_val_loss = vl / vt
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        flag = ' ← best'
    print(f'Ep {epoch:02d} | train loss={tl/tt:.4f} acc={tc/tt:.3f} | val loss={vl/vt:.4f} acc={vc/vt:.3f}{flag}')

## Cell H — Per-class accuracy on val set

In [ ]:
from collections import defaultdict

model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
model.eval()
pc, pt = defaultdict(int), defaultdict(int)
with torch.no_grad():
    for X, y in val_loader:
        X, y = X.to(device), y.to(device)
        preds = model(X).argmax(1)
        for t, p in zip(y.cpu().numpy(), preds.cpu().numpy()):
            pt[t] += 1
            if t == p: pc[t] += 1

print('Per-class val accuracy:')
for i, name in enumerate(BALL_ACTION_CLASSES):
    if pt[i]:
        print(f'  {name:<28} {pc[i]/pt[i]:.2f}  ({pc[i]}/{pt[i]})')

## Cell I — Save best model to Drive

In [ ]:
import json, torch
from pathlib import Path

save_dir = Path('/content/drive/MyDrive/ezstats/runs/event_spotter_bas2025')
save_dir.mkdir(parents=True, exist_ok=True)
torch.save(best_state, str(save_dir / 'model.pt'))
(save_dir / 'classes.json').write_text(json.dumps(BALL_ACTION_CLASSES, indent=2))
print(f'Saved → {save_dir}')
print()
print('Next steps:')
print('  1. Download model.pt → artifacts/training/event_spotter_bas2025/model.pt')
print('  2. Update event_spotter.py: SOCCERNET_CLASSES list + head size (18→14)')
print('  3. Update run_full_pipeline.ps1 + run_new_video.ps1:')
print('     --event-model-name artifacts/training/event_spotter_bas2025/model.pt')